# Kalman Filter Pairs Trading — China Futures

Dynamic hedge ratios via Kalman Filter with half-life mean-reversion signals.

**Data:** China futures daily closes (2020-01-02 → 2026-11-20), 33 contracts.


## Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from pykalman import KalmanFilter
import ffn
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4', '#E94F37', '#393E41']
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=COLORS)
%matplotlib inline


## Load data

In [ ]:
# Load long-format China futures data and pivot to wide (date x ticker)
raw = pd.read_csv('china_futures_2020_2026.csv')

# Rename columns to English
raw = raw.rename(columns={
    '日期': 'date',
    '开盘价': 'open',
    '最高价': 'high',
    '最低价': 'low',
    '收盘价': 'close',
    '成交量': 'volume',
    '持仓量': 'open_interest',
    '动态结算价': 'settlement'
})

raw['date'] = pd.to_datetime(raw['date'])

# Pivot close prices
data = raw.pivot(index='date', columns='ticker', values='close')
data = data.sort_index()

# Handle missing data: forward-fill then drop any remaining all-NaN rows
data = data.ffill().bfill()
data = data.dropna(how='all')

print(f"Shape: {data.shape}")
print(f"Date range: {data.index[0].date()} → {data.index[-1].date()}")
print(f"Tickers ({len(data.columns)}): {sorted(data.columns.tolist())}")
print(f"Missing after fill: {data.isna().sum().sum()}")
data.tail(3)


## Find cointegrated pairs

In [ ]:
def find_cointegrated_pairs(dataframe, critical_level=0.05):
    n = dataframe.shape[1]
    keys = dataframe.columns
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            try:
                score, pvalue, _ = coint(dataframe.iloc[:, i], dataframe.iloc[:, j])
                if pvalue < critical_level:
                    pairs.append([keys[i], keys[j], round(pvalue, 6)])
            except Exception:
                continue
    return pd.DataFrame(pairs, columns=['Asset1', 'Asset2', 'p-value']).sort_values('p-value')

pairs = find_cointegrated_pairs(data)
print(f"Found {len(pairs)} cointegrated pairs (p < 0.05)")
pairs.head(15)


## Create functions

In [ ]:
def half_life(spread):
    spread_lag = spread.shift(1).bfill()
    spread_ret = spread - spread_lag
    spread_lag2 = sm.add_constant(spread_lag)
    try:
        model = sm.OLS(spread_ret, spread_lag2).fit()
        hl = int(round(-np.log(2) / model.params.iloc[1], 0))
        return max(hl, 1)
    except Exception:
        return 20

def KalmanFilterAverage(x):
    kf = KalmanFilter(
        transition_matrices=[1],
        observation_matrices=[1],
        initial_state_mean=0,
        initial_state_covariance=1,
        observation_covariance=1,
        transition_covariance=0.01
    )
    state_means, _ = kf.filter(x.values)
    return pd.Series(state_means.flatten(), index=x.index)

def KalmanFilterRegression(x, y):
    delta = 1e-3
    trans_cov = delta / (1 - delta) * np.eye(2)
    obs_mat = np.expand_dims(np.vstack([[x], [np.ones(len(x))]]).T, axis=1)
    kf = KalmanFilter(
        n_dim_obs=1, n_dim_state=2,
        initial_state_mean=[0, 0],
        initial_state_covariance=np.ones((2, 2)),
        transition_matrices=np.eye(2),
        observation_matrices=obs_mat,
        observation_covariance=2,
        transition_covariance=trans_cov
    )
    state_means, _ = kf.filter(y.values)
    return state_means

def backtest(s1, s2, x, y, entry=2.0, exit=0.0):
    df = pd.DataFrame({'y': y, 'x': x}).dropna()
    if len(df) < 50:
        return pd.Series(dtype=float), 0.0, 20

    try:
        state_means = KalmanFilterRegression(
            KalmanFilterAverage(df['x']),
            KalmanFilterAverage(df['y'])
        )
    except Exception:
        return pd.Series(dtype=float), 0.0, 20

    df['hr'] = -state_means[:, 0]
    df['spread'] = df['y'] + df['x'] * df['hr']

    hl = half_life(df['spread'])
    mean = df['spread'].rolling(hl).mean()
    std = df['spread'].rolling(hl).std()
    df['z'] = (df['spread'] - mean) / std

    # Long
    df['long_entry'] = (df['z'] < -entry) & (df['z'].shift(1) >= -entry)
    df['long_exit'] = (df['z'] > -exit) & (df['z'].shift(1) <= -exit)
    df['num_long'] = np.nan
    df.loc[df['long_entry'], 'num_long'] = 1
    df.loc[df['long_exit'], 'num_long'] = 0
    df['num_long'] = df['num_long'].ffill().fillna(0)

    # Short
    df['short_entry'] = (df['z'] > entry) & (df['z'].shift(1) <= entry)
    df['short_exit'] = (df['z'] < exit) & (df['z'].shift(1) >= exit)
    df['num_short'] = np.nan
    df.loc[df['short_entry'], 'num_short'] = -1
    df.loc[df['short_exit'], 'num_short'] = 0
    df['num_short'] = df['num_short'].ffill().fillna(0)

    df['position'] = df['num_long'] + df['num_short']
    denom = (df['x'].abs() * df['hr'].abs() + df['y']).replace(0, np.nan)
    df['spread_pct'] = df['spread'].diff() / denom
    df['rets'] = df['spread_pct'] * df['position'].shift(1)
    df['cumrets'] = (1 + df['rets'].fillna(0)).cumprod()

    sharpe = 0.0
    if df['rets'].std() > 0:
        sharpe = df['rets'].mean() / df['rets'].std() * np.sqrt(252)

    return df['cumrets'], sharpe, hl


## Train / test split

In [ ]:
# ~70/30 split
split = int(len(data) * 0.70)
train = data.iloc[:split]
test = data.iloc[split:]
print(f"Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} days)")
print(f"Test : {test.index[0].date()} → {test.index[-1].date()} ({len(test)} days)")


## Choose pairs for out-sample testing

In [ ]:
# Select pairs with Sharpe > 0.5 and final equity > 1.105
list_sect = []
ret = pd.DataFrame()

for _, row in pairs.iterrows():
    s1, s2 = row['Asset1'], row['Asset2']
    name = f"{s1}-{s2}"
    try:
        tmp, sharpe, hl = backtest(s1, s2, train[s1], train[s2])
        if len(tmp) == 0:
            continue
        if sharpe > 0.5 and tmp.iloc[-1] > 1.105:
            ret[name] = tmp
            list_sect.append((s1, s2, round(sharpe, 3), hl, round(tmp.iloc[-1], 3)))
    except Exception:
        continue

print(f"Pairs selected: {len(list_sect)}")
pd.DataFrame(list_sect, columns=['s1', 's2', 'Sharpe', 'HalfLife', 'Final']).head(10)


## In-sample backtesting

In [ ]:
if len(ret) > 0:
    ret = ret.dropna(how='all')
    ret.iloc[0] = 1
    print("In-sample equity curves ready")
    ret.tail(3)
else:
    print("No pairs met selection criteria")


In [ ]:
# In-sample cumulative returns
if len(ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    ret.plot(ax=ax, linewidth=1.6, colormap='Set2')
    ax.set_title('In-Sample Cumulative Returns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Return')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_returns.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Performance stats
if len(ret) > 0:
    perf = ret.calc_stats()
    perf.display()


In [ ]:
# In-sample drawdowns
if len(ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    ffn.to_drawdown_series(ret).plot(ax=ax, linewidth=1.4, colormap='Set2')
    ax.set_title('In-Sample Drawdowns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Drawdown')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_drawdown.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Equal-weight portfolio
if len(ret) > 0:
    port = ret.mean(axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    port.plot(ax=ax, color=COLORS[0], linewidth=2.2)
    ax.set_title('In-Sample Portfolio Equity', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Return')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_portfolio.png', dpi=150, bbox_inches='tight')
    plt.show()
    stats = port.calc_stats()
    print(stats.stats[['total_return', 'cagr', 'daily_sharpe', 'max_drawdown', 'calmar']])


In [ ]:
# Portfolio drawdown
if len(ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    ffn.to_drawdown_series(port).plot(ax=ax, color=COLORS[3], linewidth=2)
    ax.set_title('In-Sample Portfolio Drawdown', fontsize=14, fontweight='bold')
    ax.set_ylabel('Drawdown')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('insample_port_drawdown.png', dpi=150, bbox_inches='tight')
    plt.show()


## Out-sample backtesting

In [ ]:
# Out-sample on selected pairs
test_ret = pd.DataFrame()
for s1, s2, *_ in list_sect:
    name = f"{s1}-{s2}"
    try:
        cum, sharpe, hl = backtest(s1, s2, test[s1], test[s2])
        if len(cum) > 0:
            test_ret[name] = cum
    except Exception:
        continue

if len(test_ret) > 0:
    test_ret.iloc[0] = 1
    print(f"Out-sample pairs: {test_ret.shape[1]}")
    test_ret.tail(3)
else:
    print("No out-sample results")


In [ ]:
# Out-sample cumulative returns
if len(test_ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    test_ret.plot(ax=ax, linewidth=1.6, colormap='Set2')
    ax.set_title('Out-Sample Cumulative Returns', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Return')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outsample_returns.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Out-sample portfolio
if len(test_ret) > 0:
    port_out = test_ret.mean(axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    port_out.plot(ax=ax, color=COLORS[1], linewidth=2.2)
    ax.set_title('Out-Sample Portfolio Equity', fontsize=14, fontweight='bold')
    ax.set_ylabel('Cumulative Return')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outsample_portfolio.png', dpi=150, bbox_inches='tight')
    plt.show()
    stats = port_out.calc_stats()
    print(stats.stats[['total_return', 'cagr', 'daily_sharpe', 'max_drawdown', 'calmar']])


In [ ]:
# Out-sample portfolio drawdown
if len(test_ret) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    ffn.to_drawdown_series(port_out).plot(ax=ax, color=COLORS[3], linewidth=2)
    ax.set_title('Out-Sample Portfolio Drawdown', fontsize=14, fontweight='bold')
    ax.set_ylabel('Drawdown')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outsample_drawdown.png', dpi=150, bbox_inches='tight')
    plt.show()
